# Exploratory Data Analysis — RACE Dataset
**Intelligent Reading Comprehension and Quiz Generation System**  
BS (CS) Spring 2026 — AI Lab Project  
Student IDs: i230538 / i230607

## 0. Imports & Setup

In [ ]:
import os
import re
import string
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# ── Path to dev.csv (single-file RACE split used in this project) ──
DATA_PATH = os.path.join('..', 'data', 'raw', 'dev.csv')
print('Data path:', DATA_PATH, '| Exists:', os.path.exists(DATA_PATH))

## 1. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 2. Basic Statistics

In [ ]:
print('=== Summary Statistics ===')
print(f'Total rows (Q-A pairs) : {len(df):,}')
print(f'Missing values per col :\n{df.isnull().sum()}')
print()

# Answer label distribution
ans_counts = df['answer'].value_counts().sort_index()
print('Answer distribution:')
print(ans_counts)
print(f'Balance check — most frequent: {ans_counts.max()/len(df)*100:.1f}% | least: {ans_counts.min()/len(df)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Answer balance bar chart
axes[0].bar(ans_counts.index, ans_counts.values,
            color=['#4e79a7','#f28e2b','#59a14f','#e15759'], edgecolor='black')
axes[0].set_title('Answer Label Distribution', fontweight='bold')
axes[0].set_xlabel('Correct Answer')
axes[0].set_ylabel('Count')
for i, v in enumerate(ans_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=10)

# Pie chart
axes[1].pie(ans_counts.values, labels=ans_counts.index,
            autopct='%1.1f%%', colors=['#4e79a7','#f28e2b','#59a14f','#e15759'],
            startangle=90)
axes[1].set_title('Answer Label Proportions', fontweight='bold')

plt.tight_layout()
plt.savefig('answer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 3. Text Length Analysis

In [ ]:
# Compute lengths in words
for col in ['article', 'question', 'A', 'B', 'C', 'D']:
    df[f'{col}_len'] = df[col].fillna('').apply(lambda x: len(str(x).split()))

# Summary table
length_stats = pd.DataFrame({
    'Column': ['article', 'question', 'A', 'B', 'C', 'D'],
    'Mean Words': [df[f'{c}_len'].mean() for c in ['article','question','A','B','C','D']],
    'Median Words': [df[f'{c}_len'].median() for c in ['article','question','A','B','C','D']],
    'Max Words': [df[f'{c}_len'].max() for c in ['article','question','A','B','C','D']],
    'Min Words': [df[f'{c}_len'].min() for c in ['article','question','A','B','C','D']],
}).set_index('Column').round(1)

print('=== Length Statistics (in words) ===')
print(length_stats.to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

cols = ['article', 'question', 'A', 'B', 'C', 'D']
colors = ['#4e79a7','#f28e2b','#59a14f','#e15759','#76b7b2','#edc948']

for ax, col, color in zip(axes, cols, colors):
    data = df[f'{col}_len'].clip(upper=df[f'{col}_len'].quantile(0.99))
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean={data.mean():.0f}')
    ax.set_title(f'{col.capitalize()} Length (words)', fontweight='bold')
    ax.set_xlabel('Word Count')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Word Length Distributions Across All Fields', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Question Type Analysis

In [ ]:
WH_WORDS = ['what', 'who', 'where', 'when', 'why', 'how', 'which']

def classify_question(q):
    q_lower = str(q).lower()
    for wh in WH_WORDS:
        if wh in q_lower.split()[:5]:  # check first 5 words
            return wh.capitalize()
    if '_____' in q_lower or '___' in q_lower:
        return 'Fill-in-blank'
    return 'Other'

df['q_type'] = df['question'].apply(classify_question)
q_type_counts = df['q_type'].value_counts()

print('=== Question Type Distribution ===')
print(q_type_counts)
print(f'\nFill-in-blank proportion: {(df["q_type"]=="Fill-in-blank").mean()*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(q_type_counts.index, q_type_counts.values,
               color=sns.color_palette('Set2', len(q_type_counts)),
               edgecolor='black')
for bar in bars:
    width = bar.get_width()
    ax.text(width + 20, bar.get_y() + bar.get_height()/2,
            f'{width:,}', va='center', fontsize=10)
ax.set_title('Question Type Distribution in RACE Dataset', fontweight='bold', fontsize=13)
ax.set_xlabel('Count')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('question_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Vocabulary & Word Frequency

In [ ]:
STOPWORDS = set([
    'the','a','an','is','was','are','were','it','he','she','they',
    'i','in','on','at','to','of','and','or','but','for','with',
    'his','her','that','this','be','have','had','has','from','as',
    'by','not','we','you','their','its','which','who','but','if',
])

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return [w for w in text.split() if w not in STOPWORDS and len(w) > 2]

# Sample 3000 articles for speed
sample = df.sample(min(3000, len(df)), random_state=42)
all_words = [w for art in sample['article'] for w in tokenize(art)]
word_freq = Counter(all_words)

print(f'Unique non-stop words in sample: {len(word_freq):,}')
print('Top 20 content words:')
for word, count in word_freq.most_common(20):
    print(f'  {word:20s}: {count:,}')

In [ ]:
top_words = word_freq.most_common(20)
words, counts = zip(*top_words)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(words, counts, color=sns.color_palette('Blues_d', 20), edgecolor='white')
ax.set_title('Top 20 Content Words in RACE Articles', fontweight='bold', fontsize=13)
ax.set_xlabel('Word')
ax.set_ylabel('Frequency')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Article Length vs Answer Label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot of article length per answer label
groups = [df[df['answer']==l]['article_len'].clip(upper=600) for l in ['A','B','C','D']]
axes[0].boxplot(groups, labels=['A','B','C','D'],
                patch_artist=True,
                boxprops=dict(facecolor='#4e79a7', alpha=0.7))
axes[0].set_title('Article Length by Correct Answer', fontweight='bold')
axes[0].set_xlabel('Correct Answer Label')
axes[0].set_ylabel('Article Length (words)')

# Option length comparison (A B C D average)
opt_means = [df[f'{l}_len'].mean() for l in ['A','B','C','D']]
axes[1].bar(['A','B','C','D'], opt_means,
             color=['#4e79a7','#f28e2b','#59a14f','#e15759'], edgecolor='black')
axes[1].set_title('Average Option Length (A/B/C/D)', fontweight='bold')
axes[1].set_xlabel('Option')
axes[1].set_ylabel('Mean Word Count')

plt.tight_layout()
plt.savefig('length_vs_answer.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Word Overlap Analysis (Article ↔ Options)

In [ ]:
def word_overlap(text_a, text_b):
    set_a = set(tokenize(str(text_a)))
    set_b = set(tokenize(str(text_b)))
    return len(set_a & set_b) / (len(set_a) + 1)

sample2 = df.sample(min(1000, len(df)), random_state=0)

overlaps = {'correct': [], 'distractor': []}
for _, row in sample2.iterrows():
    correct = row['answer']
    for opt in ['A','B','C','D']:
        ov = word_overlap(row['article'], row[opt])
        if opt == correct:
            overlaps['correct'].append(ov)
        else:
            overlaps['distractor'].append(ov)

print(f"Correct option avg overlap with article   : {np.mean(overlaps['correct']):.4f}")
print(f"Distractor avg overlap with article       : {np.mean(overlaps['distractor']):.4f}")
print(f"Difference                                : {np.mean(overlaps['correct'])-np.mean(overlaps['distractor']):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(overlaps['correct'],    bins=40, alpha=0.6, label='Correct Option',
        color='#59a14f', edgecolor='white')
ax.hist(overlaps['distractor'], bins=40, alpha=0.6, label='Distractor Options',
        color='#e15759', edgecolor='white')
ax.axvline(np.mean(overlaps['correct']),    color='green', linestyle='--',
           label=f'Correct mean={np.mean(overlaps["correct"]):.3f}')
ax.axvline(np.mean(overlaps['distractor']), color='red', linestyle='--',
           label=f'Distractor mean={np.mean(overlaps["distractor"]):.3f}')
ax.set_title('Word Overlap: Correct vs Distractor Options with Article',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Overlap Ratio (article ∩ option / |article|)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('overlap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary Statistics Table

In [ ]:
summary = {
    'Metric': [
        'Total Q-A pairs',
        'Mean Article Length (words)',
        'Median Article Length (words)',
        'Mean Question Length (words)',
        'Mean Option Length (words)',
        'Answer A proportion',
        'Answer B proportion',
        'Answer C proportion',
        'Answer D proportion',
        'Most common Q-type',
        'Avg correct-option overlap with article',
        'Avg distractor overlap with article',
    ],
    'Value': [
        f"{len(df):,}",
        f"{df['article_len'].mean():.1f}",
        f"{df['article_len'].median():.1f}",
        f"{df['question_len'].mean():.1f}",
        f"{(df['A_len'].mean()+df['B_len'].mean()+df['C_len'].mean()+df['D_len'].mean())/4:.1f}",
        f"{(df['answer']=='A').mean()*100:.1f}%",
        f"{(df['answer']=='B').mean()*100:.1f}%",
        f"{(df['answer']=='C').mean()*100:.1f}%",
        f"{(df['answer']=='D').mean()*100:.1f}%",
        df['q_type'].value_counts().idxmax(),
        f"{np.mean(overlaps['correct']):.4f}",
        f"{np.mean(overlaps['distractor']):.4f}",
    ]
}

summary_df = pd.DataFrame(summary)
print('=== FINAL EDA SUMMARY TABLE ===')
print(summary_df.to_string(index=False))

summary_df.to_csv('eda_summary.csv', index=False)
print('\nSaved to eda_summary.csv')

## 9. Key Insights

1. **Answer Balance**: The RACE dataset is roughly balanced across A/B/C/D (~25% each), confirming it is a fair benchmark for multi-class classification.
2. **Article Length**: Articles average ~300 words; some extend past 600. Preprocessing must handle length variation.
3. **Question Types**: Fill-in-blank questions dominate, followed by 'What' questions. This is expected from school-exam data.
4. **Overlap Signal**: Correct options have slightly higher word-overlap with the article than distractors — this is the key signal TF-IDF + cosine similarity exploits for answer verification.
5. **Option Length**: Answer options A–D have similar average lengths (~10 words), ruling out trivial length-based shortcuts.
6. **Vocabulary**: The dataset spans diverse topics (science, history, culture), making it a robust benchmark.